# P2 · N5a — Memorization Probe

**Paper 2 — MARQ-Bench**

Method: Bordt, Nori & Caruana, *Elephants Never Forget: Testing Language Models
for Memorization of Tabular Data*, arXiv:2403.06644 (2024).

**What matters is the contrast.** C1–C3 are public UCI corpora and should show
memorisation. C4 (NYC TLC, May 2026) post-dates every training cutoff and should
not.

---

### Results already on Drive are never regenerated

Every corpus × model × test result is cached to `memorization/` as soon as it
succeeds. On any later run, cached tests are skipped and **no API call is made
for them**. You can re-run this notebook top to bottom at any time without
spending anything on work already done.

Consequently, to add a model you set `MODELS` to that model alone. The summary
in §10 reads **every result file on disk**, not only the models in `MODELS`, so
the final artefact still contains all models regardless of which one this run
touched.

To deliberately redo a test, delete its JSON from `memorization/`.

**Section 7 has a `CLEAR_OLD` switch that deletes cached results.** It defaults
to `False` and should stay there. It exists only for the one-time repair of the
corrupted first run and is documented below.

---

**Expected:** C4's `first_token` fails with
`ValueError: Failed to construct valid first tokens`. That is correct — C4's
leading column `VendorID` is near-binary, so no discriminative token exists.
Report it as *test not applicable to C4*.

**Runtime:** minutes if everything is cached; ~30–45 min for one new model.

## 1 · Install, then RESTART

Run this, then **Runtime → Restart session**, then run from the top. The install
is skipped on the second pass.

In [ ]:
need = []
try:
    import tabmemcheck
except ImportError:
    need.append('tabmemcheck')
try:
    import anthropic
except ImportError:
    need.append('anthropic')
try:
    from google import genai
except ImportError:
    need.append('google-genai')

if need:
    print('installing:', need)
    %pip install -q {' '.join(need)}
    print('\nINSTALLED. Now: Runtime > Restart session, then run from the top.')
else:
    import tabmemcheck as _t
    print('tabmemcheck', _t.__version__, '- all dependencies present, continue')

tabmemcheck 0.1.6 - all dependencies present, continue


## 2 · Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3 · Setup

In [ ]:
import sys, os, json, datetime
from pathlib import Path
import pandas as pd

ROOT      = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES   = ROOT / 'Notebooks'
ARTIFACTS = ROOT / 'artifacts'
PROBE_DIR = ROOT / 'memorization'

assert ROOT.exists() and MODULES.exists()
for d in (ARTIFACTS, PROBE_DIR):
    d.mkdir(parents=True, exist_ok=True)
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_census as C
import tabmemcheck as tabmem
from tabmemcheck.llm import LLM_Interface

PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}

def discover_data():
    files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv','.parquet','.xlsx')]
    out = {}
    for corpus, frags in PATTERNS.items():
        hits = [p for p in files if any(f in p.name.lower() for f in frags)]
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
        if hits: out[corpus] = hits[0]
    return out

DATA_PATHS = discover_data()
print('tabmemcheck', tabmem.__version__, '| census', C.CENSUS_VERSION)
print('corpora found:', list(DATA_PATHS))

tabmemcheck 0.1.6 | census 1.0.0
corpora found: ['bank_marketing', 'diabetes_130us', 'online_retail_ii', 'nyc_tlc_yellow']


## 4 · What is already cached

Read this before setting `MODELS` in §8. Anything marked complete costs nothing
to re-run and will be skipped.

In [ ]:
TESTS = ['feature_names', 'header', 'row_completion', 'first_token']

cached = {}
for f in sorted(PROBE_DIR.glob('*__*.json')):
    r = json.loads(f.read_text())
    key = (r.get('_model', '?'), r.get('_corpus', '?'))
    cached[key] = {t: ('ok' if t in r else
                       ('failed' if t + '_error' in r else '-')) for t in TESTS}

if not cached:
    print('nothing cached yet — this is a first run')
else:
    print(f'{"model":<32}{"corpus":<18}' + ''.join(f'{t[:9]:>11}' for t in TESTS))
    print('-' * 96)
    for (m, c), st in sorted(cached.items()):
        print(f'{m:<32}{c:<18}' + ''.join(f'{st[t]:>11}' for t in TESTS))
    done_models = sorted({m for m, _ in cached})
    print(f'\nmodels with cached results: {done_models}')
    print('Set MODELS in section 8 to ONLY the models missing from this list.')

model                           corpus              feature_n     header  row_compl  first_tok
------------------------------------------------------------------------------------------------
claude-sonnet-4-5-20250929      bank_marketing             ok         ok         ok         ok
claude-sonnet-4-5-20250929      diabetes_130us             ok         ok         ok         ok
claude-sonnet-4-5-20250929      nyc_tlc_yellow             ok         ok     failed          -
claude-sonnet-4-5-20250929      online_retail_ii           ok         ok         ok         ok

models with cached results: ['claude-sonnet-4-5-20250929']
Set MODELS in section 8 to ONLY the models missing from this list.


## 5 · API keys (Colab Secrets — key icon in the sidebar)

In [ ]:
from google.colab import userdata
for name in ('ANTHROPIC_API_KEY', 'GOOGLE_API_KEY'):
    try:
        os.environ[name] = userdata.get(name)
        print(f'{name:<22} loaded')
    except Exception:
        print(f'{name:<22} not set (fine if you are not using that provider)')

ANTHROPIC_API_KEY      loaded
GOOGLE_API_KEY         loaded


## 6 · Adapters

`tabmemcheck` ships an OpenAI adapter and a Gemini one. Anthropic needs a short
adapter; `LLM_Interface` is a two-method abstraction so it is brief. The system
message is routed to Anthropic's top-level parameter, where it belongs.

In [ ]:
class AnthropicLLM(LLM_Interface):
    chat_mode = True

    def __init__(self, model_id, api_key=None):
        import anthropic
        self.model_id = model_id
        self._anthropic = anthropic
        self._client = anthropic.Anthropic(
            api_key=api_key or os.environ.get('ANTHROPIC_API_KEY'))
        self._temperature_ok = None
        self.calls = self.in_tok = self.out_tok = 0

    def __repr__(self):
        return self.model_id

    def reset_counters(self):
        self.calls = self.in_tok = self.out_tok = 0

    def chat_completion(self, messages, temperature: float, max_tokens: int):
        system_parts = [m['content'] for m in messages if m['role'] == 'system']
        convo = [{'role': m['role'], 'content': m['content']}
                 for m in messages if m['role'] != 'system']
        kw = dict(model=self.model_id, max_tokens=max_tokens, messages=convo)
        if system_parts:
            kw['system'] = '\n'.join(system_parts)
        if self._temperature_ok is not False:
            kw['temperature'] = temperature
        try:
            resp = self._client.messages.create(**kw)
        except self._anthropic.BadRequestError as exc:
            if 'temperature' not in str(exc).lower() or self._temperature_ok is False:
                raise
            self._temperature_ok = False
            kw.pop('temperature', None)
            resp = self._client.messages.create(**kw)
        else:
            if self._temperature_ok is None:
                self._temperature_ok = True
        self.calls += 1
        self.in_tok  += getattr(resp.usage, 'input_tokens', 0) or 0
        self.out_tok += getattr(resp.usage, 'output_tokens', 0) or 0
        return ''.join(b.text for b in resp.content
                       if getattr(b, 'type', '') == 'text')


class GeminiLLM(LLM_Interface):
    chat_mode = True

    class DailyQuotaExhausted(RuntimeError):
        pass

    def __init__(self, model_id, api_key=None, requests_per_minute=8):
        from google import genai
        self.model_id = model_id
        self._client = genai.Client(
            api_key=api_key or os.environ.get('GOOGLE_API_KEY'))
        self.min_interval = 60.0 / max(requests_per_minute, 0.1)
        self._last = 0.0
        self.calls = self.in_tok = self.out_tok = 0

    def __repr__(self):
        return self.model_id

    def reset_counters(self):
        self.calls = self.in_tok = self.out_tok = 0

    def chat_completion(self, messages, temperature: float, max_tokens: int):
        import time, re
        from google.genai import types

        system = '\n'.join(m['content'] for m in messages if m['role'] == 'system')
        contents = '\n'.join(m['content'] for m in messages if m['role'] != 'system')

        cfg = types.GenerateContentConfig(
            system_instruction=system or None,
            temperature=temperature,
            max_output_tokens=max_tokens,
        )

        for attempt in range(1, 8):                 # more attempts: 503s cluster
            wait = self.min_interval - (time.time() - self._last)
            if wait > 0:
                time.sleep(wait)
            self._last = time.time()
            try:
                resp = self._client.models.generate_content(
                    model=self.model_id, contents=contents, config=cfg)
                break
            except Exception as exc:
                msg = str(exc)
                is_429 = '429' in msg or 'RESOURCE_EXHAUSTED' in msg.upper()
                is_503 = '503' in msg or 'UNAVAILABLE' in msg.upper()
                if not (is_429 or is_503):
                    raise

                if is_503:
                    delay = min(self.min_interval * 2 ** attempt, 60.0)
                    print(f'  [gemini] 503 model busy, waiting {delay:.0f}s '
                          f'({attempt}/7)')
                    time.sleep(delay)
                    continue

                if 'PerDay' in msg or 'per day' in msg.lower():
                    raise self.DailyQuotaExhausted(
                        'Gemini daily quota exhausted. Cached results are safe; '
                        're-run tomorrow.') from exc

                m = re.search(r"retryDelay['\"]?\s*[:=]\s*['\"]?(\d+(?:\.\d+)?)", msg)
                delay = float(m.group(1)) if m else self.min_interval * 2 ** attempt
                print(f'  [gemini] rate limited, waiting {delay:.0f}s ({attempt}/7)')
                time.sleep(delay)
        else:
            raise RuntimeError('Gemini: exhausted retries (429/503)')

        self.calls += 1
        u = getattr(resp, 'usage_metadata', None)
        self.in_tok  += getattr(u, 'prompt_token_count', 0) or 0
        self.out_tok += getattr(u, 'candidates_token_count', 0) or 0
        return getattr(resp, 'text', '') or ''


# --- clear ONLY the three 503-failed tests so they re-run -------------------
import json
for cid, test in [('diabetes_130us', 'row_completion'),
                  ('online_retail_ii', 'row_completion'),
                  ('nyc_tlc_yellow', 'header')]:
    f = PROBE_DIR / f'{cid}__gemini-3.1-flash-lite.json'
    if not f.exists():
        continue
    r = json.loads(f.read_text())
    removed = r.pop(test + '_error', None)
    if removed:
        f.write_text(json.dumps(r, indent=2, default=str))
        print(f'cleared {cid} / {test} — will re-run')

backends = {'gemini-3.1-flash-lite': GeminiLLM('gemini-3.1-flash-lite',
                                               requests_per_minute=6)}
MODELS = list(backends)
print('\nNow re-run section 9. Only the cleared tests will make API calls.')

cleared diabetes_130us / row_completion — will re-run
cleared online_retail_ii / row_completion — will re-run
cleared nyc_tlc_yellow / header — will re-run

Now re-run section 9. Only the cleared tests will make API calls.


## 7 · Probe CSVs, and the CLEAR_OLD switch

Probe CSVs are written in original file order with original tokens via
`load_corpus`, so nothing is re-encoded — a probe against re-encoded data would
test memorisation of something the model never saw.

**`CLEAR_OLD` deletes every cached result.** It exists only for the one-time
repair of the run corrupted by the `tabmemcheck` `max_tokens` leak. Leave it
`False`.

In [ ]:
CLEAR_OLD = False      # LEAVE FALSE. True deletes all cached results.

PROBE_ROWS = 2000
tabmem.config.max_tokens = 1000     # undo any leak from a previous session

probe_files = {}
for cid, path in DATA_PATHS.items():
    df, _ = C.load_corpus(cid, path)
    out = PROBE_DIR / f'{cid}_probe.csv'
    df.head(PROBE_ROWS).to_csv(out, index=False)
    probe_files[cid] = out
    print(f'{cid:<18} {len(df):>9,} rows -> first {PROBE_ROWS:,} written')

if CLEAR_OLD:
    n = sum(1 for f in PROBE_DIR.glob('*__*.json'))
    for f in PROBE_DIR.glob('*__*.json'):
        f.unlink()
    print(f'\n!! CLEARED {n} cached result file(s)')
else:
    print(f'\ncached results preserved: '
          f'{len(list(PROBE_DIR.glob("*__*.json")))} file(s)')

bank_marketing        45,211 rows -> first 2,000 written
diabetes_130us       101,766 rows -> first 2,000 written
online_retail_ii   1,067,371 rows -> first 2,000 written
nyc_tlc_yellow     4,090,836 rows -> first 2,000 written

cached results preserved: 4 file(s)


## 8 · Scope

**Set `MODELS` to only the models NOT listed as cached in section 4.** Anything
already cached is skipped regardless, so listing it is harmless — but listing a
model whose provider you have no credit for will fail on the first uncached
test.

Roughly 55 calls per corpus × model.

In [ ]:
backends = {}

# --- register only what you intend to run --------------------------------
if os.environ.get('GOOGLE_API_KEY'):
    backends['gemini-3.1-flash-lite'] = GeminiLLM('gemini-3.1-flash-lite',
                                                  requests_per_minute=8)

# Already cached from an earlier run — uncomment only to add a NEW corpus.
# if os.environ.get('ANTHROPIC_API_KEY'):
#     backends['claude-sonnet-4-5-20250929'] = AnthropicLLM('claude-sonnet-4-5-20250929')
#     backends['claude-haiku-4-5-20251001']  = AnthropicLLM('claude-haiku-4-5-20251001')

MODELS  = list(backends)
CORPORA = list(probe_files)

todo = sum(1 for m in MODELS for c in CORPORA
           if not (PROBE_DIR / f'{c}__{m}.json').exists())
print(f'registered : {MODELS}')
print(f'corpora    : {CORPORA}')
print(f'cells to run (uncached): {todo}  ~= {todo*55} calls')
if todo == 0:
    print('\nEverything is cached. Section 9 will make no API calls.')

registered : ['gemini-3.1-flash-lite']
corpora    : ['bank_marketing', 'diabetes_130us', 'online_retail_ii', 'nyc_tlc_yellow']
cells to run (uncached): 4  ~= 220 calls


## 9 · Run

Cached tests are skipped without an API call. `max_tokens` is reset before and
after every test, so a failure cannot contaminate what follows.

In [ ]:
for model_id, llm in backends.items():
    for cid in CORPORA:
        out_json = PROBE_DIR / f'{cid}__{model_id}.json'
        r = json.loads(out_json.read_text()) if out_json.exists() else {}

        pending = [t for t in TESTS if t not in r]
        if not pending:
            print(f'[cached] {model_id} x {cid} — all tests present, skipping')
            continue

        print(f'\n{"="*66}\n{model_id}  x  {cid}\n{"="*66}')
        path = str(probe_files[cid])
        llm.reset_counters()

        for name, fn in [
            ('feature_names',  lambda: tabmem.feature_names_test(path, llm)),
            ('header',         lambda: tabmem.header_test(path, llm)),
            ('row_completion', lambda: tabmem.row_completion_test(path, llm, num_queries=25)),
            ('first_token',    lambda: tabmem.first_token_test(path, llm, num_queries=25)),
        ]:
            if name in r:
                print(f'  [cached] {name}')
                continue

            tabmem.config.max_tokens = 1000     # guard against the library leak
            try:
                r[name] = fn()
                r.pop(name + '_error', None)
            except Exception as e:
                msg = f'{type(e).__name__}: {str(e)[:300]}'
                r[name + '_error'] = msg
                print(f'  !! {name}: {msg}')
                stop = ('credit balance' in msg.lower()
                        or 'DailyQuotaExhausted' in msg
                        or 'insufficient' in msg.lower())
                if stop:
                    out_json.write_text(json.dumps(r, indent=2, default=str))
                    print('\n  STOPPING — provider quota or credit issue, not a code fault.')
                    print('  Everything completed so far is cached. Resume later.')
                    raise SystemExit
            finally:
                tabmem.config.max_tokens = 1000

            r['_model'], r['_corpus'] = model_id, cid
            r['_calls'], r['_input_tokens'], r['_output_tokens'] = (
                llm.calls, llm.in_tok, llm.out_tok)
            r['_run_at_utc'] = datetime.datetime.now(
                datetime.timezone.utc).isoformat(timespec='seconds')
            out_json.write_text(json.dumps(r, indent=2, default=str))

        print(f'  {llm.calls} calls, {llm.in_tok:,} in / {llm.out_tok:,} out')

[cached] gemini-3.1-flash-lite x bank_marketing — all tests present, skipping
[cached] gemini-3.1-flash-lite x diabetes_130us — all tests present, skipping
[cached] gemini-3.1-flash-lite x online_retail_ii — all tests present, skipping

gemini-3.1-flash-lite  x  nyc_tlc_yellow
  [cached] feature_names
  [cached] header
  [cached] row_completion
  !! first_token: ValueError: Failed to construct valid first tokens.
  0 calls, 0 in / 0 out


## 10 · Collect every result on disk

Reads **all** cached files, not only the models in `MODELS`, so the artefact
contains every model ever probed regardless of which one this run touched.

In [ ]:
results = {}
for f in sorted(PROBE_DIR.glob('*__*.json')):
    r = json.loads(f.read_text())
    results[f"{r.get('_model','?')}|{r.get('_corpus','?')}"] = r

print(f'{len(results)} corpus x model cells on disk\n')
print(f'{"corpus":<18}{"model":<32}{"role":<9}{"calls":>7}{"tok/call":>10}')
print('-' * 78)
for key, r in sorted(results.items(), key=lambda kv: kv[0].split('|')[::-1]):
    model, cid = key.split('|')
    role = 'CONTROL' if cid == 'nyc_tlc_yellow' else 'public'
    calls, out = r.get('_calls', 0), r.get('_output_tokens', 0)
    per = out / calls if calls else 0
    flag = '  <-- SUSPICIOUS' if 0 < per < 5 else ''
    print(f'{cid:<18}{model:<32}{role:<9}{calls:>7}{per:>10.1f}{flag}')

print('\nHealthy runs average tens of tokens per call. Near 1 means the')
print('tabmemcheck max_tokens leak recurred — discard those results.')

(ARTIFACTS / 'N5a_memorization.json').write_text(
    json.dumps(results, indent=2, default=str))
print(f'\nwrote {ARTIFACTS / "N5a_memorization.json"}')

12 corpus x model cells on disk

corpus            model                           role       calls  tok/call
------------------------------------------------------------------------------
bank_marketing    claude-haiku-4-5-20251001       public        55      79.0
bank_marketing    claude-sonnet-4-5-20250929      public        55      53.1
bank_marketing    gemini-3.1-flash-lite           public        55      59.0
diabetes_130us    claude-haiku-4-5-20251001       public        30     222.0
diabetes_130us    claude-sonnet-4-5-20250929      public        30     157.7
diabetes_130us    gemini-3.1-flash-lite           public        25     126.3
nyc_tlc_yellow    claude-haiku-4-5-20251001       CONTROL        0       0.0
nyc_tlc_yellow    claude-sonnet-4-5-20250929      CONTROL        0       0.0
nyc_tlc_yellow    gemini-3.1-flash-lite           CONTROL        0       0.0
online_retail_ii  claude-haiku-4-5-20251001       public        30     130.4
online_retail_ii  claude-sonnet-4-5-20250

In [ ]:
import json
files = sorted(PROBE_DIR.glob('*__*.json'))
print(f'{len(files)} result files in {PROBE_DIR}\n')
TESTS = ['feature_names','header','row_completion','first_token']
print(f'{"corpus":<18}{"model":<32}' + ''.join(f'{t[:9]:>12}' for t in TESTS))
print('-'*104)
for f in files:
    r = json.loads(f.read_text())
    st = ['ok' if t in r else ('FAILED' if t+'_error' in r else '-') for t in TESTS]
    print(f'{r.get("_corpus","?"):<18}{r.get("_model","?"):<32}' + ''.join(f'{s:>12}' for s in st))

12 result files in /content/drive/MyDrive/Paper2_RuleAuthorship/memorization

corpus            model                              feature_n      header   row_compl   first_tok
--------------------------------------------------------------------------------------------------------
bank_marketing    claude-haiku-4-5-20251001                 ok          ok          ok          ok
bank_marketing    claude-sonnet-4-5-20250929                ok          ok          ok          ok
bank_marketing    gemini-3.1-flash-lite                     ok          ok          ok          ok
diabetes_130us    claude-haiku-4-5-20251001                 ok          ok          ok          ok
diabetes_130us    claude-sonnet-4-5-20250929                ok          ok          ok          ok
diabetes_130us    gemini-3.1-flash-lite                     ok          ok          ok          ok
nyc_tlc_yellow    claude-haiku-4-5-20251001                 ok          ok          ok      FAILED
nyc_tlc_yellow    claude-

## 11 · Summary

C1–C3 are public: memorisation expected. C4 post-dates every training cutoff:
memorisation should be absent.

In [ ]:
rows = []
for key, r in results.items():
    model, cid = key.split('|')
    row = {'corpus': cid, 'model': model,
           'control': cid == 'nyc_tlc_yellow'}

    h = r.get('header')
    if isinstance(h, list) and len(h) >= 3:
        truth, resp = h[1], h[2]
        lcp = 0
        for a, b in zip(truth, resp):
            if a != b:
                break
            lcp += 1
        row['header_verbatim_chars'] = lcp
        row['header_pct'] = round(lcp / max(len(truth), 1), 4)

    fn = r.get('feature_names')
    if isinstance(fn, list) and len(fn) >= 3:
        t = [x.strip() for x in fn[1].split(',') if x.strip()]
        s = [x.strip() for x in fn[2].split(',') if x.strip()]
        row['feature_names_recovered'] = f'{sum(1 for x in t if x in s)}/{len(t)}'

    rc = r.get('row_completion')
    if isinstance(rc, list) and len(rc) == 2:
        truth, resp = rc
        row['row_exact'] = sum(1 for a, b in zip(truth, resp)
                               if a.strip() and a.strip() in b.strip())

    row['failed_tests'] = ', '.join(k[:-6] for k in r if k.endswith('_error')) or '-'
    rows.append(row)

summary = pd.DataFrame(rows).sort_values(['model', 'corpus'])
print(summary.to_string(index=False))
summary.to_csv(ARTIFACTS / 'N5a_memorization_summary.csv', index=False)
print(f'\nwrote {ARTIFACTS / "N5a_memorization_summary.csv"}')
print('\nHigh header_pct on C1-C3 with low values on C4 is the expected pattern.')

          corpus                      model  control  header_verbatim_chars  header_pct feature_names_recovered  row_exact failed_tests
  bank_marketing  claude-haiku-4-5-20251001    False                    491      0.2154                   13/13          0            -
  diabetes_130us  claude-haiku-4-5-20251001    False                     14      0.0085                   31/38          0            -
  nyc_tlc_yellow  claude-haiku-4-5-20251001     True                      7      0.0066                    9/15          0  first_token
online_retail_ii  claude-haiku-4-5-20251001    False                     48      0.0357                     5/6          0            -
  bank_marketing claude-sonnet-4-5-20250929    False                   1148      1.0000                   13/13          1            -
  diabetes_130us claude-sonnet-4-5-20250929    False                     37      0.0456                   38/38          0            -
  nyc_tlc_yellow claude-sonnet-4-5-20250929     